# Transformer Playground — Attention, Generation & Context

**Session 2 companion notebook.** Run each cell top-to-bottom, tweak inputs, and explore.

---

In [ ]:
# Install dependencies (run once)
%pip install -q transformers torch matplotlib seaborn numpy

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1 — Load BERT and Visualize Attention

BERT is an **encoder** Transformer. We'll use it to see how self-attention works — which words "look at" which other words.

In [ ]:
from transformers import BertTokenizer, BertModel

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased', output_attentions=True)
model.eval()

print(f"Model: BERT base (uncased)")
print(f"Layers: {model.config.num_hidden_layers}")
print(f"Attention heads per layer: {model.config.num_attention_heads}")
print(f"Hidden size: {model.config.hidden_size}")

In [ ]:
def get_attention(sentence, layer=0, head=0):
    """Extract attention weights from BERT."""
    inputs = tokenizer(sentence, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs)
    attention = outputs.attentions[layer][0, head].numpy()
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    return attention, tokens


def plot_attention(attention, tokens, title="Attention Weights", figsize=(10, 8)):
    """Plot an attention heatmap."""
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(attention, xticklabels=tokens, yticklabels=tokens,
                cmap='Blues', ax=ax, square=True)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Attended to (Keys)')
    ax.set_ylabel('Attending from (Queries)')
    plt.tight_layout()
    plt.show()

### The Flagship Example: Coreference

In the sentence *"The trophy didn't fit in the suitcase because it was too big"*, what does **"it"** refer to?

Let's see what BERT's attention tells us.

In [ ]:
sentence = "The trophy didn't fit in the suitcase because it was too big"
print(f"Sentence: {sentence}\n")

attention, tokens = get_attention(sentence, layer=0, head=0)
plot_attention(attention, tokens, title='Attention — Layer 1, Head 1')

In [ ]:
# Focus on the "it" token — what does it attend to?
it_idx = tokens.index('it')

print(f"Attention from 'it' to all other tokens:\n")
for i, (tok, score) in enumerate(zip(tokens, attention[it_idx])):
    bar = '█' * int(score * 40) + '░' * (40 - int(score * 40))
    marker = ' ◄◄◄' if tok == 'trophy' else ''
    print(f"  {tok:12s} {bar} {score:.3f}{marker}")

### The "bank" Problem — Revisited from Session 1

Word2Vec gave "bank" the **same vector** in both contexts. Let's see how attention creates **different representations**.

In [ ]:
sentences = [
    "I deposited money at the bank this morning",
    "We sat by the river bank and watched the sunset",
]

for sent in sentences:
    attn, toks = get_attention(sent, layer=5, head=0)
    plot_attention(attn, toks, title=f'Attention (Layer 6)\n"{sent}"')

In [ ]:
# Compare what "bank" attends to in each sentence
for sent in sentences:
    attn, toks = get_attention(sent, layer=5, head=0)
    bank_idx = toks.index('bank')
    print(f'\n"{sent}"')
    print(f"  'bank' (position {bank_idx}) attends most to:")
    top_indices = np.argsort(attn[bank_idx])[::-1][:5]
    for idx in top_indices:
        print(f"    {toks[idx]:12s} {attn[bank_idx][idx]:.3f}")

### Multi-Head Attention — Different Heads See Different Things

Each attention head learns to focus on a different type of relationship. Let's compare heads.

In [ ]:
sentence = "The cat sat on the mat because it was comfortable"

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for idx, (layer, head) in enumerate([(0, 0), (0, 5), (0, 11), (5, 0), (5, 5), (11, 0)]):
    attn, toks = get_attention(sentence, layer=layer, head=head)
    ax = axes[idx // 3][idx % 3]
    sns.heatmap(attn, xticklabels=toks, yticklabels=toks, cmap='Blues', ax=ax, square=True)
    ax.set_title(f'Layer {layer+1}, Head {head+1}', fontsize=11)

plt.suptitle(f'"{sentence}"', fontsize=13)
plt.tight_layout()
plt.show()

## 2 — GPT-2 Text Generation

GPT is a **decoder** Transformer. It generates text by predicting one token at a time.

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

gpt_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt_model = GPT2LMHeadModel.from_pretrained('gpt2')
gpt_model.eval()

print(f"Model: GPT-2 (small)")
print(f"Layers: {gpt_model.config.n_layer}")
print(f"Attention heads per layer: {gpt_model.config.n_head}")
print(f"Vocabulary size: {gpt_model.config.vocab_size:,}")

### Next-Token Probabilities

Before generating, let's see what GPT-2 **thinks** should come next.

In [ ]:
def show_next_token_probs(prompt, top_k=10):
    """Show the probability distribution for the next token."""
    inputs = gpt_tokenizer(prompt, return_tensors='pt')
    with torch.no_grad():
        outputs = gpt_model(**inputs)
    
    logits = outputs.logits[0, -1, :]
    probs = torch.softmax(logits, dim=0)
    top_probs, top_indices = torch.topk(probs, top_k)
    
    print(f'Prompt: "{prompt}"')
    print(f'Top {top_k} next-token predictions:\n')
    for i, (prob, idx) in enumerate(zip(top_probs, top_indices)):
        token = gpt_tokenizer.decode([idx])
        bar = '█' * int(prob.item() * 50) + '░' * (50 - int(prob.item() * 50))
        print(f'  {i+1:2d}. {token:15s} {bar} {prob.item():.3f}')

show_next_token_probs("The cat sat on the")

In [ ]:
# --- Try your own prompts! ---
show_next_token_probs("The president of the United States")

### Temperature — Controlling Randomness

**Low temperature** (0.3) → picks the most likely token → safe, repetitive  
**High temperature** (1.5) → samples more randomly → creative, sometimes incoherent

In [ ]:
# --- Try changing the prompt! ---
prompt = "Natural language processing allows computers to"

inputs = gpt_tokenizer(prompt, return_tensors='pt')

print(f'Prompt: "{prompt}"\n')
print('=' * 70)

for temp in [0.3, 1.0, 1.5]:
    with torch.no_grad():
        output = gpt_model.generate(
            **inputs,
            max_new_tokens=40,
            temperature=temp,
            do_sample=True,
            pad_token_id=gpt_tokenizer.eos_token_id,
        )
    text = gpt_tokenizer.decode(output[0], skip_special_tokens=True)
    label = {0.3: 'Conservative', 1.0: 'Balanced', 1.5: 'Creative'}[temp]
    print(f'\n[Temperature {temp} — {label}]')
    print(text)
    print('-' * 70)

In [ ]:
# Another prompt — try your own!
prompt = "Once upon a time in a land far away"
inputs = gpt_tokenizer(prompt, return_tensors='pt')

print(f'Prompt: "{prompt}"\n')
for temp in [0.3, 1.0, 1.5]:
    with torch.no_grad():
        output = gpt_model.generate(
            **inputs, max_new_tokens=40, temperature=temp,
            do_sample=True, pad_token_id=gpt_tokenizer.eos_token_id,
        )
    text = gpt_tokenizer.decode(output[0], skip_special_tokens=True)
    print(f'[T={temp}] {text}\n')

## 3 — BERT Masked Language Model

BERT (encoder) was trained to fill in **masked** words. This shows it understands context in both directions.

In [ ]:
from transformers import pipeline

filler = pipeline('fill-mask', model='bert-base-uncased')

# --- Try changing the sentence! Keep [MASK] where you want the prediction ---
sentences = [
    "The cat [MASK] on the mat",
    "I went to the [MASK] to deposit money",
    "We sat by the river [MASK] and watched the sunset",
    "Paris is the capital of [MASK]",
]

for sent in sentences:
    print(f'\n"{sent}"')
    results = filler(sent)
    for r in results[:5]:
        bar = '█' * int(r['score'] * 30)
        print(f"  {r['token_str']:12s} {bar} {r['score']:.3f}")

## 4 — Contextual Embeddings vs Static Embeddings

The key insight from Session 2: the **same word gets different vectors** depending on context.

Let's prove it by extracting BERT's hidden states for "bank" in two different sentences.

In [ ]:
def get_word_embedding(sentence, target_word, layer=-1):
    """Get BERT's contextual embedding for a word in a sentence."""
    inputs = tokenizer(sentence, return_tensors='pt')
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Find the target word's position
    target_idx = tokens.index(target_word)
    # Get the hidden state at the specified layer
    embedding = outputs.last_hidden_state[0, target_idx].numpy()
    return embedding


# Get "bank" embeddings in two different contexts
emb_finance = get_word_embedding("I deposited money at the bank", "bank")
emb_river = get_word_embedding("We sat by the river bank", "bank")
emb_sat1 = get_word_embedding("The cat sat on the mat", "sat")
emb_sat2 = get_word_embedding("She sat for the exam yesterday", "sat")

# Compute cosine similarity
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Cosine similarity between contextual embeddings:")
print(f"  bank (finance) vs bank (river):   {cosine_sim(emb_finance, emb_river):.3f}")
print(f"  sat (chair) vs sat (exam):        {cosine_sim(emb_sat1, emb_sat2):.3f}")
print(f"  bank (finance) vs sat (chair):    {cosine_sim(emb_finance, emb_sat1):.3f}")
print()
print("With static embeddings (Word2Vec), both 'bank' pairs would be similarity = 1.0")
print("With contextual embeddings, same word in different contexts → different vectors!")

## 5 — Your Turn!

Use the cells below to experiment. Change sentences, try different layers/heads, and explore.

In [ ]:
# Your attention visualization
my_sentence = "The dog chased the cat because it was fast"
attn, toks = get_attention(my_sentence, layer=0, head=0)
plot_attention(attn, toks, title=f'Your sentence: "{my_sentence}"')

In [ ]:
# Your text generation
my_prompt = "The future of artificial intelligence is"
inputs = gpt_tokenizer(my_prompt, return_tensors='pt')

with torch.no_grad():
    output = gpt_model.generate(
        **inputs, max_new_tokens=50, temperature=1.0,
        do_sample=True, pad_token_id=gpt_tokenizer.eos_token_id,
    )
print(gpt_tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
# Your masked prediction
my_masked = "The [MASK] jumped over the lazy dog"
results = filler(my_masked)
for r in results[:5]:
    print(f"  {r['token_str']:12s} {r['score']:.3f}")